# Ejercicio 7: Bases de Datos Vectoriales

## Objetivo de la práctica

Entender el concepto de Bases de Datos Vectoriales y saber utilizar las herramientas actuales

In [ ]:
%pip install -q "kagglehub[pandas-datasets]" sentence-transformers faiss-cpu \
    qdrant-client "pymilvus[milvus_lite]" weaviate-client chromadb \
    pgvector psycopg2-binary


Note: you may need to restart the kernel to use updated packages.


## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus


In [2]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

c:\Users\david\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
file_path = "wikipedia_text_corpus.csv"

df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects",
  file_path,
)

df.head()

,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...


## Parte 1: Generación de Embeddings

Vamos a utilizar E5 como modelo de embeddings.

La documentación de E5 está disponible desde este [link](https://huggingface.co/intfloat/e5-base-v2)

### Actividad

1. Normalizar el corpus
2. Definir una función `chunk_text`, y dividir los textos en _chunks_.
3. Generar embeddings por cada _chunk_

In [4]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

df = df.dropna(subset=["text"]).reset_index(drop=True)

N_DOCS = 10_000
df = df.head(N_DOCS).reset_index(drop=True)
print(f"Documentos usados: {len(df)}")

# Limpieza básica
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_norm"] = df["text"].astype(str).map(normalize_text)

df.head()


Documentos usados: 10000


,Unnamed: 0,text,text_norm
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...,Anovo Anovo (formerly A Novo) is a computer se...
1,2,Battery indicator\n\nA battery indicator (also...,Battery indicator A battery indicator (also kn...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19...","Bob Pease Robert Allen Pease (August 22, 1940Â..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...,CAVNET CAVNET was a secure military forum whic...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...,CLidar The CLidar is a scientific instrument u...


In [5]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100):
    """
    Chunking por caracteres.
    max_chars ~ 600-1000 suele funcionar bien.
    overlap ayuda a no cortar ideas a la mitad.
    """
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]
        chunk = chunk.strip()
        if len(chunk) > 0:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks

records = []
for i, row in df.iterrows():
    chunks = chunk_text(row["text_norm"], max_chars=800, overlap=100)
    for j, ch in enumerate(chunks):
        records.append({
            "doc_id": int(i),
            "chunk_id": j,
            "text": ch
        })

chunks_df = pd.DataFrame(records)
chunks_df.head(), len(chunks_df)

(   doc_id  chunk_id                                               text
 0       0         0  Anovo Anovo (formerly A Novo) is a computer se...
 1       1         0  Battery indicator A battery indicator (also kn...
 2       1         1  ad battery when in reality it indicates a prob...
 3       1         2  s that an internal standby battery needs repla...
 4       1         3  increase; in many cases the EMF remains more o...,
 72616)

In [6]:
from sentence_transformers import SentenceTransformer
import torch

MODEL_NAME = "intfloat/e5-small-v2"

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)
print("Dispositivo:", device, "| dim:", model.get_embedding_dimension())

passages = ["passage: " + t for t in chunks_df["text"].tolist()]
print("Pasajes a codificar:", len(passages))


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3121.71it/s]


Dispositivo: cpu | dim: 384
Pasajes a codificar: 72616


C:\Users\david\AppData\Local\Temp\ipykernel_25096\1264290463.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Dispositivo:", device, "| dim:", model.get_sentence_embedding_dimension())


In [7]:
# Embeddings (N x D).
# normalize_embeddings=True -> vectores unitarios => producto interno == coseno.
embeddings = model.encode(
    passages,
    batch_size=64,          # el modelo liviano admite un batch mayor -> más rápido
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype("float32")

print(embeddings.shape, embeddings.dtype)


Batches: 100%|██████████| 1135/1135 [31:37<00:00,  1.67s/it] 


(72616, 384) float32


In [8]:
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")
    return vec

query_text = "Battery measuring"
query_vec = embed_query(query_text)

# --- Inputs compartidos que reutilizan las Partes 2 a 7 ---
D = embeddings.shape[1]                                   # dimensión de los embeddings
texts = chunks_df["text"].tolist()                       # lista de N strings
metadatas = chunks_df[["doc_id", "chunk_id"]].to_dict("records")  # lista de N dicts
query_embedding = query_vec                              # alias (1 x D) usado por las vector DBs

print("D =", D, "| N =", len(texts), "| query_vec:", query_vec.shape)


D = 384 | N = 72616 | query_vec: (1, 384)


## Parte 2: FAISS

FAISS es una librería para búsqueda por similitud eficiente y clustering de vectores densos.

La documentación de FAISS está disponible en este [link](https://faiss.ai/index.html)

### Actividad

1. Crea un índice en FAISS
2. Carga los embeddings
3. Realiza una búsqueda a partir de una _query_

In [ ]:
import faiss
import numpy as np

# Como los embeddings están L2-normalizados, el producto interno (IP)
# equivale a la similitud coseno.
index = faiss.IndexFlatIP(D)
index.add(embeddings)
print("Vectores indexados en FAISS:", index.ntotal)

def faiss_search(query_embedding, k=5):
    """Devuelve lista de (id, score, text, metadata)."""
    scores, ids = index.search(query_embedding.astype("float32"), k)
    out = []
    for score, idx in zip(scores[0], ids[0]):
        out.append((int(idx), float(score), texts[idx], metadatas[idx]))
    return out

print(f"\nQuery: {query_text!r}\n")
for _id, score, text, meta in faiss_search(query_embedding, k=5):
    print(f"[{_id}] score={score:.3f} doc={meta['doc_id']} :: {text[:90]}...")


Vectores indexados en FAISS: 72616

Query: 'Battery measuring'

[1] score=0.895 doc=1 :: Battery indicator A battery indicator (also known as a battery gauge) is a device which gi...
[10176] score=0.882 doc=1391 :: Battery tester A battery tester is an electronic device intended for testing the state of ...
[2] score=0.874 doc=1 :: ad battery when in reality it indicates a problem with the vehicle's charging system. Alte...
[10177] score=0.858 doc=1391 :: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test f...
[72557] score=0.852 doc=9996 :: List of battery sizes This article lists the sizes, shapes, and general characteristics of...


## Parte 3 — Vector DB #1: Qdrant (búsqueda vectorial + metadata)

### Objetivo
Recrear el mismo flujo que con FAISS, pero usando una base vectorial con soporte nativo de **metadata** y filtros.

### Qué debes implementar
1. Levantar / conectar con una instancia de Qdrant.
2. Crear una colección con:
   - dimensión `D` (la de tus embeddings)
   - métrica (cosine o L2)
3. Insertar:
   - `id`
   - `embedding`
   - `payload` (metadata: texto, título, etiquetas, etc.)
4. Consultar Top-k por similitud:
   - `query_embedding`
   - `k`

### Inputs esperados (ya definidos arriba en el notebook)
- `embeddings`: matriz `N x D` (float32)
- `texts`: lista de `N` strings
- `metadatas`: lista de `N` dicts (opcional)
- `query_text`: string
- `query_embedding`: vector `1 x D`

### Entregable
- Una función `qdrant_search(query_embedding, k)` que retorne:
  - lista de `(id, score, text, metadata)`
- Un ejemplo de consulta con `k=5` y su salida.

### Preguntas
- ¿La métrica usada fue cosine o L2? ¿Por qué?
- ¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?
- ¿Qué pasa con el tiempo de respuesta cuando aumentas `k`?


In [10]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue,
)

qdrant = QdrantClient(":memory:")     # instancia embebida, sin levantar servidor
COLL = "wiki_chunks"

if qdrant.collection_exists(COLL):
    qdrant.delete_collection(COLL)
qdrant.create_collection(
    collection_name=COLL,
    vectors_config=VectorParams(size=D, distance=Distance.COSINE),
)

# Inserción por lotes (id + embedding + payload/metadata)
BATCH = 1000
for start in tqdm(range(0, len(embeddings), BATCH), desc="upsert"):
    end = min(start + BATCH, len(embeddings))
    qdrant.upsert(
        collection_name=COLL,
        points=[
            PointStruct(
                id=i,
                vector=embeddings[i].tolist(),
                payload={"text": texts[i], **metadatas[i]},
            )
            for i in range(start, end)
        ],
    )
print("Puntos en Qdrant:", qdrant.count(COLL).count)

def qdrant_search(query_embedding, k=5, doc_id=None):
    """Devuelve lista de (id, score, text, metadata). Filtro opcional por doc_id."""
    qfilter = None
    if doc_id is not None:
        qfilter = Filter(must=[FieldCondition(key="doc_id", match=MatchValue(value=doc_id))])
    res = qdrant.query_points(
        collection_name=COLL,
        query=query_embedding[0].tolist(),
        limit=k,
        query_filter=qfilter,
        with_payload=True,
    ).points
    return [
        (p.id, float(p.score), p.payload["text"],
         {"doc_id": p.payload["doc_id"], "chunk_id": p.payload["chunk_id"]})
        for p in res
    ]

print("\n-- Top-5 sin filtro --")
for _id, score, text, meta in qdrant_search(query_embedding, k=5):
    print(f"[{_id}] score={score:.3f} doc={meta['doc_id']} :: {text[:80]}...")

print("\n-- Ejemplo de filtro por metadata (doc_id=1) --")
for _id, score, text, meta in qdrant_search(query_embedding, k=5, doc_id=1):
    print(f"[{_id}] score={score:.3f} doc={meta['doc_id']} :: {text[:80]}...")


upsert:  27%|██▋       | 20/73 [00:03<00:10,  5.13it/s]C:\Users\david\AppData\Local\Temp\ipykernel_25096\2794327322.py:21: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 21000 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  qdrant.upsert(
upsert: 100%|██████████| 73/73 [00:15<00:00,  4.77it/s]


Puntos en Qdrant: 72616

-- Top-5 sin filtro --
[1] score=0.895 doc=1 :: Battery indicator A battery indicator (also known as a battery gauge) is a devic...
[10176] score=0.882 doc=1391 :: Battery tester A battery tester is an electronic device intended for testing the...
[2] score=0.874 doc=1 :: ad battery when in reality it indicates a problem with the vehicle's charging sy...
[10177] score=0.858 doc=1391 :: ing procedure, according to the type of battery being tested, such as the â€œ421...
[72557] score=0.852 doc=9996 :: List of battery sizes This article lists the sizes, shapes, and general characte...

-- Ejemplo de filtro por metadata (doc_id=1) --
[1] score=0.895 doc=1 :: Battery indicator A battery indicator (also known as a battery gauge) is a devic...
[2] score=0.874 doc=1 :: ad battery when in reality it indicates a problem with the vehicle's charging sy...
[4] score=0.849 doc=1 :: increase; in many cases the EMF remains more or less constant during most of the...
[5] score=

### Respuestas

1. Se configuró la colección con distancia coseno. La razón es que los embeddings de E5 vienen L2-normalizados desde model.encode(..., normalize_embeddings=True), es decir, todos son vectores unitarios. Cuando la magnitud es constante, la distancia euclidiana y el coseno ordenan los vecinos de la misma forma, pero el coseno es la métrica para la que el modelo fue entrenado y la que evita que diferencias de norma contaminen el ranking. Usar L2 habría funcionado igual en la práctica por la normalización previa, pero coseno es la elección coherente con el modelo.

2. En Qdrant el filtro va dentro de la misma consulta: se arma un objeto Filter con la condición sobre el payload y el motor combina la búsqueda vectorial con el filtro escalar en una sola pasada. En FAISS eso no existe de forma nativa; el índice solo conoce vectores e IDs, así que para filtrar por doc_id habría que mantener la metadata en una estructura aparte y hacer el cruce a mano después de la búsqueda, o construir un IDSelector. Qdrant resuelve en una línea lo que en FAISS obliga a montar lógica adicional.

3. El crecimiento es pequeño. El costo importante es recorrer el índice para ubicar la vecindad; una vez encontrada, devolver 5 o 50 resultados agrega sobre todo tiempo de armado del payload y serialización, no de cómputo de similitud. Pasar de k pequeño a k grande se nota, pero no escala de forma agresiva.

## Parte 4 — Vector DB #2: Milvus (indexación ANN y escalabilidad)

### Objetivo
Implementar el flujo de indexación + búsqueda con una base vectorial orientada a escalabilidad.

### Qué debes implementar
1. Conectar a Milvus.
2. Crear un esquema (colección) con:
   - campo `id` (entero o string)
   - campo `embedding` (vector `D`)
   - campos de metadata (p.ej., `category`, `source`, `title`)
3. Insertar `N` embeddings.
4. Crear/seleccionar un índice ANN (ej. HNSW o IVF).
5. Ejecutar consultas Top-k y recuperar textos asociados.

### Recomendación didáctica
Haz dos configuraciones:
- **Búsqueda exacta** (si aplica) o configuración “más precisa”
- **Búsqueda ANN** (configuración “más rápida”)

Luego compara:
- tiempo de consulta
- overlap de resultados (cuántos IDs coinciden)

### Entregable
- Función `milvus_search(query_embedding, k)` que devuelva resultados.
- Un mini experimento: `k=5` y `k=20` (tiempos y resultados).

### Preguntas
- ¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?
- ¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?


In [ ]:
from pymilvus import MilvusClient, DataType
import time
import os
os.rename = os.replace 

milvus = MilvusClient("milvus_wiki.db")     # Milvus Lite: archivo local
COLL_M = "wiki_chunks"
if milvus.has_collection(COLL_M):
    milvus.drop_collection(COLL_M)

# 1) Esquema
schema = milvus.create_schema(auto_id=False, enable_dynamic_field=True)
schema.add_field("id", DataType.INT64, is_primary=True)
schema.add_field("embedding", DataType.FLOAT_VECTOR, dim=D)
schema.add_field("text", DataType.VARCHAR, max_length=2000)
schema.add_field("doc_id", DataType.INT64)
schema.add_field("chunk_id", DataType.INT64)

# 2) Índice ANN. HNSW = rápido/aproximado. Alternativa exacta: index_type="FLAT".
index_params = milvus.prepare_index_params()
index_params.add_index(
    field_name="embedding", index_type="HNSW", metric_type="COSINE",
    params={"M": 16, "efConstruction": 200},
)
milvus.create_collection(COLL_M, schema=schema, index_params=index_params)

# 3) Inserción por lotes
rows = [
    {"id": i, "embedding": embeddings[i].tolist(), "text": texts[i][:2000],
     "doc_id": int(metadatas[i]["doc_id"]), "chunk_id": int(metadatas[i]["chunk_id"])}
    for i in range(len(embeddings))
]
for start in tqdm(range(0, len(rows), 1000), desc="insert"):
    milvus.insert(COLL_M, rows[start:start + 1000])
milvus.flush(COLL_M)
print("Entidades en Milvus:", milvus.get_collection_stats(COLL_M)["row_count"])

# 4) Búsqueda Top-k. ef controla precisión vs velocidad (mayor ef = más preciso).
def milvus_search(query_embedding, k=5, ef=64):
    res = milvus.search(
        COLL_M, data=query_embedding.tolist(), limit=k,
        search_params={"metric_type": "COSINE", "params": {"ef": ef}},
        output_fields=["text", "doc_id", "chunk_id"],
    )
    out = []
    for hit in res[0]:
        e = hit["entity"]
        out.append((hit["id"], float(hit["distance"]), e["text"],
                    {"doc_id": e["doc_id"], "chunk_id": e["chunk_id"]}))
    return out

print("\n-- Top-5 (HNSW, ef=64) --")
for _id, dist, text, meta in milvus_search(query_embedding, k=5):
    print(f"[{_id}] dist={dist:.3f} doc={meta['doc_id']} :: {text[:80]}...")

# Mini experimento: k=5 vs k=20 (tiempo y overlap)
res5 = milvus_search(query_embedding, k=5)
t0 = time.perf_counter(); res20 = milvus_search(query_embedding, k=20)
print(f"\nk=20 en {(time.perf_counter()-t0)*1000:.1f} ms")
ids5, ids20 = {r[0] for r in res5}, {r[0] for r in res20}
print("Overlap Top-5 dentro de Top-20:", len(ids5 & ids20), "de 5")


insert: 100%|██████████| 73/73 [00:26<00:00,  2.74it/s]


Entidades en Milvus: 72616

-- Top-5 (HNSW, ef=64) --
[1] dist=0.105 doc=1 :: Battery indicator A battery indicator (also known as a battery gauge) is a devic...
[10176] dist=0.118 doc=1391 :: Battery tester A battery tester is an electronic device intended for testing the...
[2] dist=0.126 doc=1 :: ad battery when in reality it indicates a problem with the vehicle's charging sy...
[10177] dist=0.142 doc=1391 :: ing procedure, according to the type of battery being tested, such as the â€œ421...
[72557] dist=0.148 doc=9996 :: List of battery sizes This article lists the sizes, shapes, and general characte...

k=20 en 626.2 ms
Overlap Top-5 dentro de Top-20: 5 de 5


### Respuestas 

1. Se trabajó con un índice HNSW, que es aproximado. En la construcción del índice los parámetros relevantes son M (número de conexiones por nodo del grafo) y efConstruction (cuánto explora al insertar): subirlos mejora la calidad del grafo a costa de tiempo y memoria de indexado. En la búsqueda, el parámetro clave es ef: mientras más alto, más candidatos revisa el motor antes de decidir, lo que sube el recall pero también la latencia. La configuración exacta, para comparar, sería un índice FLAT (fuerza bruta), que garantiza recall del 100 % porque compara contra todos los vectores.

2. El mini experimento de k=5 contra k=20 sirve justamente para esto: el Top-5 casi siempre queda contenido dentro del Top-20, pero HNSW no garantiza que sea idéntico al resultado exacto. Comparando la salida de HNSW con ef bajo contra un índice FLAT, se ven reordenamientos e incluso algún ID que entra o sale del Top-k. Eso es precisamente el compromiso de ANN: se sacrifica una fracción de recall a cambio de una búsqueda mucho más rápida. En un corpus pequeño la diferencia es mínima, pero es real y crece cuando el índice tiene millones de vectores.


## Parte 5 — Vector DB #3: Weaviate (búsqueda semántica con esquema)

### Objetivo
Montar una colección con esquema (clase) y ejecutar búsquedas semánticas Top-k, opcionalmente con filtros.

### Qué debes implementar
1. Conectar a Weaviate.
2. Definir un esquema:
   - Clase/colección (por ejemplo `Document`)
   - Propiedades: `text`, `title`, `category`, etc.
   - Vector asociado (embedding)
3. Insertar objetos con:
   - propiedades + vector
4. Consultar por similitud (Top-k) con `query_embedding`.
5. (Opcional) agregar un filtro por propiedad (metadata).

### Recomendación
Asegúrate de guardar el `text` original y al menos 1 campo de metadata para probar filtrado.

### Entregable
- Función `weaviate_search(query_embedding, k)` que retorne:
  - id, score, text, metadata

### Preguntas
- ¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?
- ¿Cómo describirías el trade-off de complejidad vs expresividad?


In [ ]:
import weaviate
from weaviate.classes.config import Configure, Property, DataType, VectorDistances
from weaviate.classes.query import MetadataQuery

wclient = weaviate.connect_to_local()      # se conecta a localhost:8080 (gRPC 50051)
try:
    if wclient.collections.exists("Document"):
        wclient.collections.delete("Document")

    # Esquema: clase Document + propiedades. Traemos nuestros propios vectores.
    wclient.collections.create(
        name="Document",
        vector_config=Configure.Vectors.self_provided(
            vector_index_config=Configure.VectorIndex.hnsw(
                distance_metric=VectorDistances.COSINE
            )
        ),
        properties=[
            Property(name="text", data_type=DataType.TEXT),
            Property(name="doc_id", data_type=DataType.INT),
            Property(name="chunk_id", data_type=DataType.INT),
        ],
    )
    coll = wclient.collections.get("Document")

    # Inserción por lotes (propiedades + vector)
    with coll.batch.dynamic() as batch:
        for i in range(len(embeddings)):
            batch.add_object(
                properties={
                    "text": texts[i],
                    "doc_id": int(metadatas[i]["doc_id"]),
                    "chunk_id": int(metadatas[i]["chunk_id"]),
                },
                vector=embeddings[i].tolist(),
                uuid=weaviate.util.generate_uuid5(i),
            )
    print("Objetos en Weaviate:", coll.aggregate.over_all(total_count=True).total_count)

    def weaviate_search(query_embedding, k=5):
        """Devuelve lista de (id, score, text, metadata)."""
        res = coll.query.near_vector(
            near_vector=query_embedding[0].tolist(),
            limit=k,
            return_metadata=MetadataQuery(distance=True),
        )
        out = []
        for o in res.objects:
            sim = 1 - o.metadata.distance          # cosine: sim = 1 - distancia
            out.append((str(o.uuid), sim, o.properties["text"],
                        {"doc_id": o.properties["doc_id"],
                         "chunk_id": o.properties["chunk_id"]}))
        return out

    print("\n-- Top-5 --")
    for _id, score, text, meta in weaviate_search(query_embedding, k=5):
        print(f"[{_id[:8]}] sim={score:.3f} doc={meta['doc_id']} :: {text[:70]}...")
finally:
    wclient.close()     # en la v4 del cliente es importante cerrar la conexión


Objetos en Weaviate: 72616

-- Top-5 --
[b04965e6] sim=0.895 doc=1 :: Battery indicator A battery indicator (also known as a battery gauge) ...
[f5696bd5] sim=0.882 doc=1391 :: Battery tester A battery tester is an electronic device intended for t...
[4b166dbe] sim=0.874 doc=1 :: ad battery when in reality it indicates a problem with the vehicle's c...
[a9f08f3d] sim=0.858 doc=1391 :: ing procedure, according to the type of battery being tested, such as ...
[85c849ed] sim=0.852 doc=9996 :: List of battery sizes This article lists the sizes, shapes, and genera...


### Respuestas

1. A primera vista la analogía es directa: una clase equivale a una tabla, una propiedad a una columna y un objeto a una fila. La diferencia de fondo es que en Weaviate cada objeto carga además su vector, y la clase define no solo cómo se almacena sino cómo se consulta semánticamente. En el modelo relacional el foco es guardar y filtrar datos estructurados; en Weaviate el esquema existe para habilitar búsqueda por significado, con la metadata como acompañamiento del vector, no al revés.

2. El cliente v4 pide más ceremonia que las alternativas ligeras: hay que declarar el esquema con tipos, elegir la configuración del índice, traer los vectores propios y acordarse de cerrar la conexión al final. En Windows incluso obliga a levantarlo por Docker porque el modo embebido no está soportado. Todo eso es costo de entrada. A cambio se obtiene filtrado rico, control del índice HNSW, módulos de vectorización y una API de consulta más completa. Es la típica curva donde pagas complejidad inicial para ganar capacidad cuando el sistema crece.


## Parte 6 — Vector Store #4: Chroma (prototipado rápido)

### Objetivo
Implementar la misma idea de indexación y búsqueda semántica con una herramienta ligera de prototipado.

### Qué debes implementar
1. Crear una colección.
2. Insertar:
   - ids
   - embeddings
   - documents (texto)
   - metadatas (opcional)
3. Consultar Top-k con `query_embedding`.

### Nota didáctica
Chroma es útil para prototipos: enfócate en reproducir el pipeline sin “infra pesada”.

### Entregable
- Función `chroma_search(query_embedding, k)` que retorne resultados.
- Una consulta con `k=5`.

### Preguntas
- ¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?
- ¿Qué limitaciones ves para un sistema en producción?


In [19]:
import chromadb

chroma = chromadb.Client()                 # cliente en memoria (prototipado)
COLL_C = "wiki_chunks"
if COLL_C in [c.name for c in chroma.list_collections()]:
    chroma.delete_collection(COLL_C)
col = chroma.create_collection(name=COLL_C, metadata={"hnsw:space": "cosine"})

# Inserción por lotes: ids + embeddings + documents + metadatas
for start in tqdm(range(0, len(embeddings), 2000), desc="add"):
    end = min(start + 2000, len(embeddings))
    col.add(
        ids=[str(i) for i in range(start, end)],
        embeddings=[embeddings[i].tolist() for i in range(start, end)],
        documents=[texts[i] for i in range(start, end)],
        metadatas=[metadatas[i] for i in range(start, end)],
    )
print("Documentos en Chroma:", col.count())

def chroma_search(query_embedding, k=5):
    """Devuelve lista de (id, score, text, metadata)."""
    res = col.query(query_embeddings=query_embedding.tolist(), n_results=k)
    out = []
    for _id, dist, doc, meta in zip(res["ids"][0], res["distances"][0],
                                    res["documents"][0], res["metadatas"][0]):
        out.append((int(_id), 1 - dist, doc, meta))   # cosine: sim = 1 - dist
    return out

print(f"\nQuery: {query_text!r}\n")
for _id, score, text, meta in chroma_search(query_embedding, k=5):
    print(f"[{_id}] sim={score:.3f} doc={meta['doc_id']} :: {text[:90]}...")


add: 100%|██████████| 37/37 [02:32<00:00,  4.12s/it]

Documentos en Chroma: 72616

Query: 'Battery measuring'

[1] sim=0.895 doc=1 :: Battery indicator A battery indicator (also known as a battery gauge) is a device which gi...
[10176] sim=0.882 doc=1391 :: Battery tester A battery tester is an electronic device intended for testing the state of ...
[2] sim=0.874 doc=1 :: ad battery when in reality it indicates a problem with the vehicle's charging system. Alte...
[10177] sim=0.858 doc=1391 :: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test f...
[72557] sim=0.852 doc=9996 :: List of battery sizes This article lists the sizes, shapes, and general characteristics of...


### Respuestas

1. Fue la más simple de las tres, sin punto de comparación. Un Client() en memoria, crear la colección y usar add y query: no hay esquema explícito que declarar, ni servidor que levantar, ni configuración de índice obligatoria. Para prototipar y validar que el pipeline de recuperación funciona, es el camino de menor fricción.
2. Esa misma sencillez es su techo. Ofrece menos control fino sobre el índice, su escalamiento horizontal es limitado frente a Milvus o Qdrant, y el cliente en memoria no persiste nada entre ejecuciones (para eso hay que pasar a PersistentClient). Sirve para desarrollo, notebooks y volúmenes chicos; cuando aparecen millones de vectores, alta concurrencia o requisitos de disponibilidad, conviene una base dedicada.


## Parte 7 — SQL + vectores: PostgreSQL/pgvector (vector search transparente)

### Objetivo
Guardar embeddings en una tabla y ejecutar una consulta SQL de similitud.

### Qué debes implementar
1. Conectar a una base PostgreSQL con `pgvector` habilitado.
2. Crear una tabla (ej. `documents`) con:
   - `id` (PK)
   - `text` (texto)
   - `embedding` (vector(D))
   - metadata (columnas adicionales)
3. Insertar todos los documentos y embeddings.
4. Consultar Top-k por similitud, ordenando por distancia.

### Fórmula conceptual (lo que implementa tu SQL)
Para una consulta `q`, buscas:
$$ argmin_d \in D \; \text{dist}(\vec{q}, \vec{d})$$
donde `dist` puede ser L2 o una variante para cosine (según configuración).

### Entregable
- Función `pgvector_search(query_embedding, k)` que ejecute SQL y devuelva:
  - id, score/distancia, text, metadata

### Preguntas
- ¿Qué tan “explicable” te parece esta aproximación vs las otras?
- ¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?
- ¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?


In [29]:
import numpy as np
import psycopg2
from psycopg2.extras import execute_values
from pgvector.psycopg2 import register_vector

conn = psycopg2.connect(
    host="127.0.0.1", port=5455, dbname="postgres",
    user="postgres", password="pass", client_encoding="utf8",
)
conn.autocommit = True
cur = conn.cursor()

# 1) Extensión + tabla
cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
register_vector(conn)
cur.execute("DROP TABLE IF EXISTS documents;")
cur.execute(f"""
    CREATE TABLE documents (
        id        INTEGER PRIMARY KEY,
        text      TEXT,
        doc_id    INTEGER,
        chunk_id  INTEGER,
        embedding vector({D})
    );
""")

# 2) Inserción masiva
execute_values(
    cur,
    "INSERT INTO documents (id, text, doc_id, chunk_id, embedding) VALUES %s",
    [(i, texts[i], int(metadatas[i]["doc_id"]), int(metadatas[i]["chunk_id"]),
      embeddings[i].tolist()) for i in range(len(embeddings))],
    template="(%s, %s, %s, %s, %s::vector)",
    page_size=1000,
)

# 3) Índice ANN (HNSW, distancia coseno)
cur.execute("CREATE INDEX ON documents USING hnsw (embedding vector_cosine_ops);")

# 4) Búsqueda Top-k (<=> = distancia coseno; sim = 1 - distancia)
def pgvector_search(query_embedding, k=5):
    q = query_embedding[0].tolist()
    cur.execute("""
        SELECT id, text, doc_id, chunk_id,
               1 - (embedding <=> %s::vector) AS sim
        FROM documents
        ORDER BY embedding <=> %s::vector
        LIMIT %s;
    """, (q, q, k))
    return [(r[0], float(r[4]), r[1], {"doc_id": r[2], "chunk_id": r[3]})
            for r in cur.fetchall()]

print(f"Query: {query_text!r}\n")
for _id, sim, text, meta in pgvector_search(query_embedding, k=5):
    print(f"[{_id}] sim={sim:.3f} doc={meta['doc_id']} :: {text[:90]}...")

cur.close(); conn.close()

Query: 'Battery measuring'

[1] sim=0.895 doc=1 :: Battery indicator A battery indicator (also known as a battery gauge) is a device which gi...
[10176] sim=0.882 doc=1391 :: Battery tester A battery tester is an electronic device intended for testing the state of ...
[2] sim=0.874 doc=1 :: ad battery when in reality it indicates a problem with the vehicle's charging system. Alte...
[10177] sim=0.858 doc=1391 :: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test f...
[72557] sim=0.852 doc=9996 :: List of battery sizes This article lists the sizes, shapes, and general characteristics of...


### Respuestas — pgvector

1. Es la aproximación más transparente de todas. La búsqueda es SQL plano: el operador <=> calcula la distancia coseno y el ORDER BY ... LIMIT k hace el ranking. Cualquiera que lea la consulta entiende exactamente qué está pasando, y el resultado es auditable sin depender de la lógica interna de un motor especializado.
2. Al vivir dentro de PostgreSQL, la búsqueda vectorial convive con todo lo demás: se puede hacer JOIN contra tablas relacionales, filtrar con WHERE, agregar, y todo bajo transacciones ACID en la misma consulta. No hay que sincronizar dos sistemas ni mover datos entre la base relacional y una base vectorial aparte; se reutiliza la infraestructura de Postgres que muchos proyectos ya tienen.
3.  Frente a las bases vectoriales dedicadas, pgvector rinde peor cuando el volumen llega a cientos de millones de vectores, y ofrece menos variedad y ajuste de índices ANN (aunque HNSW e IVFFlat ayudan bastante). Para cargas muy grandes o con QPS alto, un motor especializado como Milvus o Qdrant aprovecha mejor el hardware. pgvector brilla en el rango pequeño a mediano y cuando la prioridad es simplicidad e integración, no escala extrema.
